# Aula 20 — Interpretabilidade: coeficientes, permutation importance e SHAP

Laboratório reproduzível sobre três perguntas:

1. como escala e colinearidade afetam coeficientes?
2. quanto um modelo depende de features individuais e de um grupo correlacionado?
3. como o *background* altera Shapley sem alterar a previsão?

**Hipóteses registradas antes da execução**

- coeficientes individuais das cópias correlacionadas serão instáveis, mas sua soma será estável;
- permutation importance individual dividirá crédito, enquanto permutar o grupo mostrará dependência maior;
- Shapley reconstruirá exatamente a previsão, mas baseline e atribuições mudarão entre referências.

Os dados são sintéticos; o split é feito antes das análises e toda importância global é medida no teste reservado.

## Ambiente

Dependências mínimas: Python 3.10, NumPy 1.26, Matplotlib 3.8 e scikit-learn 1.4. Não usamos o pacote SHAP: para quatro features, enumerar as \(2^4=16\) coalizões torna a função de valor auditável.

```python
# Em um Colab novo, se necessário:
# %pip install "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4"
```

In [ ]:
import platform
import warnings
from itertools import combinations
from math import factorial

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("error")
SEED = 20260908
rng = np.random.default_rng(SEED)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("seed:", SEED)

## 1. Dados e protocolo

A unidade é uma observação independente. O processo contém um sinal, uma cópia quase perfeita, um segundo sinal registrado em escala cem vezes maior e uma feature sem efeito. O alvo usa o sinal latente e o contexto antes da mudança de unidade. Separamos 30% para teste.

In [ ]:
n = 2_200
signal = rng.normal(size=n)
signal_proxy = signal + rng.normal(0, 0.035, size=n)
context = rng.normal(size=n)
pure_noise = rng.normal(size=n)
y = 3.0 * signal + 1.2 * context + rng.normal(0, 0.55, size=n)

feature_names = np.array(["sinal", "sinal_proxy", "contexto_x100", "ruido"])
X = np.column_stack((signal, signal_proxy, 100 * context, pure_noise))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)
print("treino/teste:", X_train.shape, X_test.shape)
print("correlação sinal–proxy:", round(np.corrcoef(X_train[:, 0], X_train[:, 1])[0, 1], 6))

## 2. Coeficientes: unidade e representação

Comparamos coeficientes em unidades brutas e após StandardScaler. As previsões OLS são equivalentes; muda a unidade do parâmetro.

In [ ]:
linear_raw = LinearRegression().fit(X_train, y_train)
linear_std = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
raw_coef = linear_raw.coef_
std_coef = linear_std.named_steps["linearregression"].coef_
linear_test_r2 = linear_raw.score(X_test, y_test)

print("R² no teste:", round(linear_test_r2, 6))
for name, raw, std in zip(feature_names, raw_coef, std_coef):
    print(f"{name:14s} bruto={raw: .6f} | padronizado={std: .6f}")

O coeficiente bruto de contexto_x100 é pequeno porque uma unidade da coluna representa \(0{,}01\) unidade do contexto original. Isso não torna a feature irrelevante. Agora reamostramos apenas o treino 200 vezes.

In [ ]:
bootstrap_coef = []
for _ in range(200):
    sample_idx = rng.integers(0, len(X_train), size=len(X_train))
    fitted = LinearRegression().fit(X_train[sample_idx], y_train[sample_idx])
    bootstrap_coef.append(fitted.coef_)

bootstrap_coef = np.asarray(bootstrap_coef)
individual_sd = bootstrap_coef[:, :2].std(axis=0)
sum_signal_coef = bootstrap_coef[:, :2].sum(axis=1)
sum_sd = sum_signal_coef.std()
print("DP coeficientes sinal/proxy:", np.round(individual_sd, 6))
print("média da soma:", round(sum_signal_coef.mean(), 6))
print("DP da soma:", round(sum_sd, 6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(bootstrap_coef[:, 0], bins=24, alpha=0.65, label="sinal")
axes[0].hist(bootstrap_coef[:, 1], bins=24, alpha=0.65, label="proxy")
axes[0].set(title="Crédito individual instável", xlabel="coeficiente", ylabel="frequência")
axes[0].legend()
axes[1].hist(sum_signal_coef, bins=24, color="#2a9d8f")
axes[1].set(title="Soma estável", xlabel="coeficiente sinal + proxy")
plt.tight_layout()
plt.show()

**Leitura:** as duas colunas são substituíveis e OLS redistribui crédito. O mecanismo usa coeficiente total 3; a soma se concentra perto dele. Uma narrativa baseada apenas em qual cópia recebeu maior coeficiente seria frágil.

## 3. Permutation importance individual

Treinamos uma Random Forest e validamos seu \(R^2\) no teste antes da explicação. Cada coluna é permutada 20 vezes no mesmo teste. A importância é queda de \(R^2\), podendo exceder 1 se o score corrompido ficar negativo.

In [ ]:
forest = RandomForestRegressor(
    n_estimators=250,
    min_samples_leaf=3,
    max_features=1.0,
    random_state=SEED,
    n_jobs=1,
).fit(X_train, y_train)

forest_test_r2 = forest.score(X_test, y_test)
perm = permutation_importance(
    forest, X_test, y_test,
    scoring="r2",
    n_repeats=20,
    random_state=SEED,
    n_jobs=1,
)
print("R² da floresta no teste:", round(forest_test_r2, 6))
for name, mean, sd in zip(feature_names, perm.importances_mean, perm.importances_std):
    print(f"{name:14s} queda={mean: .6f} ± {sd:.6f}")

In [ ]:
order = np.argsort(perm.importances_mean)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh(
    feature_names[order],
    perm.importances_mean[order],
    xerr=perm.importances_std[order],
    color="#457b9d",
    alpha=0.85,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="queda de R² no teste", title="Permutation importance individual")
plt.tight_layout()
plt.show()

## 4. Permutação agrupada

Para sinal + proxy, aplicamos a mesma permutação de linhas às duas colunas. Preservamos a relação entre elas e quebramos sua associação conjunta com o alvo.

In [ ]:
group_rng = np.random.default_rng(SEED + 1)
group_drops = []
for _ in range(20):
    row_perm = group_rng.permutation(len(X_test))
    X_corrupted = X_test.copy()
    X_corrupted[:, [0, 1]] = X_test[row_perm][:, [0, 1]]
    group_drops.append(forest_test_r2 - forest.score(X_corrupted, y_test))

group_drops = np.asarray(group_drops)
individual_signal_sum = perm.importances_mean[:2].sum()
print("soma das quedas individuais:", round(individual_signal_sum, 6))
print(f"queda agrupada: {group_drops.mean():.6f} ± {group_drops.std():.6f}")

As quedas individuais não são aditivas: cada perturbação define outro dataset e a redundância permite substituição. O grupo mede dependência da família, não causalidade.

## 5. Shapley exato com função de valor explícita

Usamos a definição interventional:

\[
v(S)=\frac{1}{|B|}\sum_{z\in B}f(x_S,z_{\bar S}),
\]

onde \(B\) é o *background*. Copiamos suas linhas e substituímos as colunas da coalizão pelos valores da observação. Para quatro features, armazenamos 16 valores.

In [ ]:
def coalition_value(model, x, background, coalition):
    hybrid = background.copy()
    if coalition:
        cols = list(coalition)
        hybrid[:, cols] = x[cols]
    return float(model.predict(hybrid).mean())


def exact_interventional_shapley(model, x, background):
    n_features = x.shape[0]
    values = {}
    for size in range(n_features + 1):
        for coalition in combinations(range(n_features), size):
            key = frozenset(coalition)
            values[key] = coalition_value(model, x, background, key)

    phi = np.zeros(n_features)
    for j in range(n_features):
        others = [k for k in range(n_features) if k != j]
        for size in range(n_features):
            for coalition in combinations(others, size):
                S = frozenset(coalition)
                weight = (
                    factorial(size)
                    * factorial(n_features - size - 1)
                    / factorial(n_features)
                )
                phi[j] += weight * (values[S | {j}] - values[S])
    return values[frozenset()], phi, values

Escolhemos uma observação de previsão alta. O background A usa 250 linhas do teste; o B representa o segmento com sinal maior que 0,7. Modelo e observação permanecem idênticos.

In [ ]:
test_prediction = forest.predict(X_test)
observation_idx = int(np.argmax(test_prediction))
x_explain = X_test[observation_idx]
prediction = float(test_prediction[observation_idx])

background_general = X_test[:250]
background_high_signal = X_test[X_test[:, 0] > 0.7][:250]

base_general, phi_general, values_general = exact_interventional_shapley(
    forest, x_explain, background_general
)
base_segment, phi_segment, values_segment = exact_interventional_shapley(
    forest, x_explain, background_high_signal
)
reconstructed_general = base_general + phi_general.sum()
reconstructed_segment = base_segment + phi_segment.sum()

print("previsão explicada:", round(prediction, 6))
print("background geral:", len(background_general), "baseline", round(base_general, 6))
print("phi geral:", dict(zip(feature_names.tolist(), np.round(phi_general, 6).tolist())))
print("background segmento:", len(background_high_signal), "baseline", round(base_segment, 6))
print("phi segmento:", dict(zip(feature_names.tolist(), np.round(phi_segment, 6).tolist())))
print("erros de reconstrução:", abs(prediction - reconstructed_general), abs(prediction - reconstructed_segment))

In [ ]:
positions = np.arange(len(feature_names))
width = 0.38
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(positions - width / 2, phi_general, width, label="background geral")
ax.bar(positions + width / 2, phi_segment, width, label="segmento sinal alto")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(
    xticks=positions,
    xticklabels=feature_names,
    ylabel="contribuição para a saída",
    title="Mesma previsão, referências diferentes",
)
ax.legend()
plt.tight_layout()
plt.show()

O baseline do segmento é maior porque já contém observações com sinal elevado. As atribuições de sinal e proxy diminuem em relação a essa referência. A previsão permanece a mesma.

## 6. Verificações automáticas

Os asserts verificam desempenho, escala, instabilidade individual, estabilidade conjunta, dependência agrupada, ruído, 16 coalizões e eficiência local nos dois backgrounds.

In [ ]:
assert linear_test_r2 > 0.95
assert abs(std_coef[2]) > 50 * abs(raw_coef[2])
assert individual_sd.min() > 10 * sum_sd
assert abs(sum_signal_coef.mean() - 3.0) < 0.05
assert forest_test_r2 > 0.93
assert abs(perm.importances_mean[3]) < 0.02
assert group_drops.mean() > 1.5 * individual_signal_sum
assert len(values_general) == 16 and len(values_segment) == 16
assert abs(prediction - reconstructed_general) < 1e-10
assert abs(prediction - reconstructed_segment) < 1e-10
assert abs(base_general - base_segment) > 2.0

print("Todas as verificações passaram.")
print({
    "r2_linear_teste": round(linear_test_r2, 6),
    "dp_coef_individual_min": round(float(individual_sd.min()), 6),
    "dp_soma_coef": round(float(sum_sd), 6),
    "r2_floresta_teste": round(forest_test_r2, 6),
    "perm_sinal": round(float(perm.importances_mean[0]), 6),
    "perm_proxy": round(float(perm.importances_mean[1]), 6),
    "perm_grupo": round(float(group_drops.mean()), 6),
    "baseline_geral": round(base_general, 6),
    "baseline_segmento": round(base_segment, 6),
    "erro_shapley_max": float(max(
        abs(prediction - reconstructed_general),
        abs(prediction - reconstructed_segment),
    )),
})

## Conclusões

- Coeficiente é condicional à unidade e à representação.
- Colinearidade tornou o crédito individual instável sem destruir a previsão nem a soma do efeito.
- Permutation importance mediu dependência no teste; o grupo revelou substituição entre cópias.
- Shapley satisfez eficiência até precisão numérica.
- Trocar o *background* alterou baseline e narrativa, não a previsão.

Os resultados descrevem este modelo e este processo sintético. Não demonstram causalidade, justiça ou segurança. Em produção, registre versão, população, output, perturbação, incerteza e mecanismo de contestação.

## Próxima aula

Na [Aula 21](../aulas/21-clustering-kmeans-hierarquico-dbscan.md), estudaremos agrupamento sem target. A ausência de rótulo torna ainda mais importante não transformar um padrão geométrico em categoria substantiva sem estabilidade e validação de domínio.